# pytorch で深層学習　スカラー重みづけ Attention

各記述子(特徴量)に **サンプルごとのスカラー重み** $a_j(x)$ を掛けてから MLP に入力する。<BR>
1) 通常の MLP (baseline) と Attention 付き MLP の予測精度を比較<BR>
2) 重み $a_j$ から、記述子の重要度(全体平均)と、サンプルごとの動的な重要度を解析<BR>

$$a(x)=D\cdot\mathrm{softmax}\bigl(W_2\,\mathrm{ReLU}(W_1 x+b_1)+b_2\bigr),\quad \tilde{x}_j=a_j(x)\,x_j$$

softmax の合計を D 倍して、平均的な重みが 1 になるようにしている(重み=1 が「通常の MLP と同じ」)。

In [ ]:
#! pip install torch
! wget -nc https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/chubu-dx2026/main/oxide_refractive_index.no_outliers.csv

In [ ]:
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
torch.manual_seed(0); np.random.seed(0)

## 1. データ準備(DNN2 と同じ)

記述子のスケールが大きく異なるため、**train で fit した StandardScaler** で規格化する。attention の softmax が特徴量のスケールに引きずられないためにも必要。

In [ ]:
#df = pd.read_csv("./oxide_refractive_index.no_outliers.csv", index_col=0).dropna()
path=str("./oxide_refractive_index.no_outliers.csv")
df_original=pd.read_csv(path, index_col=0)
df=df_original.dropna()

drop_columns = ['refractive_index','composition']
df_descriptor = df.drop(drop_columns, axis=1)
target_columns = ['refractive_index']
df_target = df[target_columns]
feature_names = np.array(df_descriptor.columns)

X = df_descriptor.values
t = df_target.values
train_X, test_X, train_t, test_t = train_test_split(X, t, test_size=0.3, random_state=0)

scaler = StandardScaler().fit(train_X)
train_Xs = scaler.transform(train_X)
test_Xs  = scaler.transform(test_X)

to_t = lambda a: torch.tensor(a, dtype=torch.float32).to(device)
train_X, test_X, train_t, test_t = map(to_t, (train_Xs, test_Xs, train_t, test_t))
D = train_X.shape[1]
print(train_X.shape, test_X.shape)

## 2. モデル定義

`ScalarAttentionNet` は `(予測値, 重み a)` を返す。baseline は同じ MLP 本体を attention 無しで使う。

In [ ]:
def make_mlp(d_in, p=0.2):
    return nn.Sequential(
        nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(p),
        nn.Linear(128, 64),   nn.ReLU(), nn.Dropout(p),
        nn.Linear(64, 16),    nn.ReLU(),
        nn.Linear(16, 1),
    )


class ScalarAttentionNet(nn.Module):
    def __init__(self, d_in, d_hidden=64):
        super().__init__()
        self.score = nn.Sequential(          # x -> 各特徴量のスコア
            nn.Linear(d_in, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_in),
        )
        self.mlp = make_mlp(d_in)
        self.d_in = d_in

    def attention(self, x):
        return torch.softmax(self.score(x), dim=1) * self.d_in   # 平均 1

    def forward(self, x):
        a = self.attention(x)
        return self.mlp(x * a), a

baseline = make_mlp(D).to(device)
attn_net = ScalarAttentionNet(D).to(device)

## 3. 学習

損失は MSE。attention 側には任意でエントロピー正則化 `lam_ent` を付けられる(大きくすると重みが少数の特徴量に集中し、解釈しやすくなる。0 なら無効)。

In [ ]:
loss_fn = nn.MSELoss()

def fit(model, use_attn, epochs=200, lr=1e-3, batch_size=64, lam_ent=0.0, wd=1e-3):  #wd = 1e-3 から減らしていく。
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(TensorDataset(train_X, train_t), batch_size=batch_size, shuffle=True)
    hist_tr, hist_te = [], []
    for epoch in tqdm(range(epochs)):
        model.train()
        tot = 0.0
        for xb, tb in loader:
            opt.zero_grad()
            if use_attn:
                yb, a = model(xb)
                p = a / a.sum(dim=1, keepdim=True)
                ent = -(p * torch.log(p + 1e-12)).sum(dim=1).mean()
                loss = loss_fn(yb, tb) + lam_ent * ent
            else:
                loss = loss_fn(model(xb), tb)
            loss.backward()
            opt.step()
            tot += float(loss) * len(xb)
        model.eval()
        with torch.no_grad():
            yt = model(test_X)
            yt = yt[0] if use_attn else yt
            hist_te.append(float(loss_fn(yt, test_t)))
        hist_tr.append(tot / len(train_X))
    return hist_tr, hist_te

h_base = fit(baseline, use_attn=False)
h_attn = fit(attn_net, use_attn=True, lam_ent=0.0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for k, (name, h) in enumerate([("baseline", h_base), ("attention", h_attn)]):
    ax[k].plot(h[0], label="train"); ax[k].plot(h[1], label="test")
    ax[k].set_title(name); ax[k].set_xlabel("epoch"); ax[k].legend()
ax[0].set_ylabel("MSE"); plt.tight_layout()

## 4. 予測能の比較

In [ ]:
def predict(model, use_attn, X):
    model.eval()
    with torch.no_grad():
        y = model(X)
        y = y[0] if use_attn else y
    return y.cpu().numpy().ravel()

yt = test_t.cpu().numpy().ravel()
rows = []
for name, m, ua in [("baseline", baseline, False), ("attention", attn_net, True)]:
    y_te = predict(m, ua, test_X)
    y_tr = predict(m, ua, train_X)
    rows.append(dict(model=name,
                     train_RMSE=mean_squared_error(train_t.cpu().numpy().ravel(), y_tr) ** 0.5,
                     test_RMSE=mean_squared_error(yt, y_te) ** 0.5,
                     test_R2=r2_score(yt, y_te)))
display(pd.DataFrame(rows).set_index("model"))

fig, ax = plt.subplots(1, 2, figsize=(8, 3.8), sharex=True, sharey=True)
for k, (name, m, ua) in enumerate([("baseline", baseline, False), ("attention", attn_net, True)]):
    ax[k].plot(predict(m, ua, train_X), train_t.cpu().numpy().ravel(), '+', alpha=.4, label="train")
    ax[k].plot(predict(m, ua, test_X), yt, 'x', alpha=.6, label="test")
    ax[k].set_title(name); ax[k].set_xlabel("predicted"); ax[k].legend()
ax[0].set_ylabel("true"); plt.tight_layout()

※ 1回の乱数・分割では差が偶然の範囲かもしれない。比較を主張するなら、`random_state` や `torch.manual_seed` を変えて複数回(または K-fold)繰り返すこと。

## 5. 重要度解析

### 5-1. 全体の重要度 = テスト集合での重み $a_j$ の平均

In [ ]:
attn_net.eval()
with torch.no_grad():
    A_test = attn_net.attention(test_X).cpu().numpy()       # (n_test, D)

imp = A_test.mean(axis=0)
top = np.argsort(imp)[::-1][:20]
plt.figure(figsize=(7, 5))
plt.barh(range(20), imp[top][::-1])
plt.yticks(range(20), feature_names[top][::-1])
plt.axvline(1.0, color='gray', ls='--', lw=1)          # 重み=1 が平均
plt.xlabel("mean attention weight"); plt.tight_layout()
pd.DataFrame({"feature": feature_names[top], "mean_a": imp[top]})

### 5-2. 動的な重要度 = サンプルごとの重みのばらつき

平均が高くても、サンプル間で変わらないなら「静的」な重要度。標準偏差が大きい特徴量ほど、入力によって重要度が切り替わる。

In [ ]:
std = A_test.std(axis=0)
top_dyn = np.argsort(std)[::-1][:20]
plt.figure(figsize=(7, 5))
plt.barh(range(20), std[top_dyn][::-1])
plt.yticks(range(20), feature_names[top_dyn][::-1])
plt.xlabel("std of attention weight over samples"); plt.tight_layout()

# 上位特徴量について、サンプルごとの重みを目的変数で色付け
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for k in range(3):
    j = top_dyn[k]
    sc = ax[k].scatter(test_Xs[:, j], A_test[:, j], c=yt, cmap="viridis", s=12)
    ax[k].set_xlabel(f"{feature_names[j]} (scaled)"); ax[k].set_ylabel("a")
plt.colorbar(sc, ax=ax, label="true MEvalues_type1")

In [ ]:
# サンプル x 上位特徴量 のヒートマップ(目的変数でソート)
order = np.argsort(yt)
plt.figure(figsize=(9, 4))
plt.imshow(A_test[order][:, top[:20]].T, aspect="auto", cmap="jet",vmin=0, vmax=5)
plt.yticks(range(20), feature_names[top[:20]], fontsize=6)
plt.xlabel("test samples (sorted by true value)"); plt.colorbar(label="a")
plt.tight_layout()